
# Séance du 29 août – Optimisation, Données déséquilibrées & Bonnes pratiques avancées

Dans cette séance, nous allons :  
1. Comparer plusieurs modèles de classification via un pipeline.  
2. Sélectionner le meilleur modèle.  
3. Optimiser ses hyperparamètres avec **GridSearchCV**.  
4. Gérer le problème de classes déséquilibrées (SMOTE, class weights).  

Le dataset utilisé est **credit.csv**, qui contient des informations de crédit et une cible binaire (got_credit oui/non).


In [1]:
# Télécharger le fichier CSV directement depuis GitHub
!wget https://raw.githubusercontent.com/bidoscar/AFRICITIZEN-ACDS-Coding/main/Seance_6_Classification/Datasets/credit.csv

--2025-09-12 18:35:58--  https://raw.githubusercontent.com/bidoscar/AFRICITIZEN-ACDS-Coding/main/Seance_6_Classification/Datasets/credit.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8251 (8.1K) [text/plain]
Saving to: ‘credit.csv’

credit.csv          100%[===================>]   8.06K  --.-KB/s    in 0s      

2025-09-12 18:35:58 (69.6 MB/s) - ‘credit.csv’ saved [8251/8251]



In [2]:
# Charger le jeu de données
import pandas as pd
df = pd.read_csv('credit.csv')

In [3]:
df.head()

,Age,Gender,Income,Education,Marital_Status,Number_of_Children,Home_Ownership,got_credit
0,25,Female,50000,Bachelor's Degree,Single,0,Rented,1
1,30,Male,100000,Master's Degree,Married,2,Owned,1
2,35,Female,75000,Doctorate,Married,1,Owned,1
3,40,Male,125000,High School Diploma,Single,0,Owned,1
4,45,Female,100000,Bachelor's Degree,Married,3,Owned,1


In [4]:
# Table des frequences de la colonne got_credit
df['got_credit'].value_counts(normalize=True)

,proportion
got_credit,
1,0.689024
0,0.310976


# **⚖️ Méthodes pour gérer les données déséquilibrées**

In [5]:
X = df.drop('got_credit', axis=1)
y = df['got_credit']

In [17]:
# Faire le split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify = y)

In [9]:
X_train.shape, X_test.shape

((131, 7), (33, 7))

In [18]:
y_train.value_counts(normalize=True)

,proportion
got_credit,
1,0.687023
0,0.312977


In [19]:
y_test.value_counts(normalize=True)

,proportion
got_credit,
1,0.69697
0,0.30303


## **🔹 1. Oversampling (sur-échantillonnage)**

👉 Idée : augmenter artificiellement la proportion de la classe minoritaire.

### **1.1 - Random Oversampling : dupliquer aléatoirement des exemples de la classe minoritaire.**

In [20]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

In [21]:
y_train_ros.value_counts(normalize=True)

,proportion
got_credit,
0,0.5
1,0.5


In [22]:
y_train_ros.shape

(180,)

### **1.2 - SMOTE (Synthetic Minority Over-sampling Technique) : génère des exemples synthétiques interpolés.**

Comment ça marche ?

* On prend un point minoritaire (ex. un client en
défaut).

* On cherche ses k plus proches voisins minoritaires (par défaut k=5).

* On génère un nouveau point synthétique en interpolant entre ce point et un voisin choisi au hasard.

## **2. Undersampling (sous-échantillonnage)**

👉 Idée : réduire la classe majoritaire pour équilibrer.

### **2.1 - Random Undersampling : supprimer des exemples de la classe majoritaire.**

In [29]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

In [44]:
y_train.value_counts()

,count
got_credit,
1,90
0,41


In [45]:
y_train_rus.value_counts()

,count
got_credit,
0,41
1,41


In [30]:
X_train_rus.shape

(82, 7)

In [32]:
y_train_rus.value_counts(normalize=True)

,proportion
got_credit,
0,0.5
1,0.5


# **3. Méthodes intégrées dans les modèles**

Ajouter un poids aux classes dans certains algorithmes :

* Logistic Regression : class_weight="balanced"

* Random Forest : class_weight="balanced"

* XGBoost/LightGBM : scale_pos_weight

# **Pipeline**

Modeliser sans aucun changement

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score

# Colonnes numeriques et categorielles
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Pipeline de pre-processing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

# Models a comparer
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(criterion = "entropy"),
    "Random Forest": RandomForestClassifier(criterion = "entropy", random_state = 42),
    "Gradient Boosting": GradientBoostingClassifier(random_state = 42),
    "SVM": SVC(kernel = "sigmoid", random_state = 42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "XGBoost": XGBClassifier(random_state = 42)
}


results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
    results[name] = scores.mean()

# Ordonner et afficher resultats
results = {k: v for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True)}
results

{'Naive Bayes': np.float64(0.9834749034749034),
 'Decision Tree': np.float64(0.9831746031746033),
 'Gradient Boosting': np.float64(0.9831746031746033),
 'SVM': np.float64(0.9831746031746033),
 'Logistic Regression': np.float64(0.9777691977691978),
 'Random Forest': np.float64(0.9777691977691978),
 'KNN': np.float64(0.9777691977691978),
 'XGBoost': np.float64(0.9726482873851294)}

Ajouter un control (en utilisant les paramètres) dans le model quand c'est possible

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score

# Colonnes numeriques et categorielles
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Pipeline de pre-processing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

# Models a comparer
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", criterion = "entropy"),
    "Random Forest": RandomForestClassifier(class_weight="balanced", criterion = "entropy", random_state = 42),
    "Gradient Boosting": GradientBoostingClassifier(random_state = 42),
    "SVM": SVC(kernel = "sigmoid", random_state = 42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "XGBoost": XGBClassifier(random_state = 42, scale_pos_weight = ratio)
}


results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
    results[name] = scores.mean()

# Ordonner et afficher resultats
results = {k: v for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True)}
results

{'Naive Bayes': np.float64(0.9834749034749034),
 'Logistic Regression': np.float64(0.9831746031746033),
 'Decision Tree': np.float64(0.9831746031746033),
 'Gradient Boosting': np.float64(0.9831746031746033),
 'SVM': np.float64(0.9831746031746033),
 'Random Forest': np.float64(0.9777691977691978),
 'KNN': np.float64(0.9777691977691978),
 'XGBoost': np.float64(0.9777691977691978)}

Refaire le modele en utilisant les données issues du Over Sampling

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score

# Colonnes numeriques et categorielles
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Pipeline de pre-processing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

# Models a comparer
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(criterion = "entropy"),
    "Random Forest": RandomForestClassifier(criterion = "entropy", random_state = 42),
    "Gradient Boosting": GradientBoostingClassifier(random_state = 42),
    "SVM": SVC(kernel = "sigmoid", random_state = 42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "XGBoost": XGBClassifier(random_state = 42)
}


results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])
    scores = cross_val_score(pipeline, X_train_ros, y_train_ros, cv=5, scoring='f1')
    results[name] = scores.mean()

# Ordonner et afficher resultats
results = {k: v for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True)}
results

{'Logistic Regression': np.float64(0.9831660231660232),
 'Decision Tree': np.float64(0.9831660231660232),
 'Random Forest': np.float64(0.9831660231660232),
 'Gradient Boosting': np.float64(0.9831660231660232),
 'SVM': np.float64(0.9831660231660232),
 'XGBoost': np.float64(0.9831660231660232),
 'KNN': np.float64(0.9723637923637923),
 'Naive Bayes': np.float64(0.9723637923637923)}

Refaire le modele en utilisant les données issues du Under Sampling

In [43]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score

# Colonnes numeriques et categorielles
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Pipeline de pre-processing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

# Models a comparer
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(criterion = "entropy"),
    "Random Forest": RandomForestClassifier(criterion = "entropy", random_state = 42),
    "Gradient Boosting": GradientBoostingClassifier(random_state = 42),
    "SVM": SVC(kernel = "sigmoid", random_state = 42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "XGBoost": XGBClassifier(random_state = 42)
}


results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])
    scores = cross_val_score(pipeline, X_train_rus, y_train_rus, cv=5, scoring='f1')
    results[name] = scores.mean()

# Ordonner et afficher resultats
results = {k: v for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True)}
results

{'Decision Tree': np.float64(0.9761403508771929),
 'Gradient Boosting': np.float64(0.9761403508771929),
 'SVM': np.float64(0.9761403508771929),
 'Logistic Regression': np.float64(0.96437564499484),
 'Random Forest': np.float64(0.96437564499484),
 'KNN': np.float64(0.96437564499484),
 'Naive Bayes': np.float64(0.96437564499484),
 'XGBoost': np.float64(0.96437564499484)}

En utilisant la méthod SMOTE

In [49]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.model_selection import cross_val_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbPipeline


# Colonnes numeriques et categorielles
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

# Pipeline de pre-processing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore',sparse_output=False), cat_cols)
    ])

# Models a comparer
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(criterion = "entropy"),
    "Random Forest": RandomForestClassifier(criterion = "entropy", random_state = 42),
    "Gradient Boosting": GradientBoostingClassifier(random_state = 42),
    "SVM": SVC(kernel = "sigmoid", random_state = 42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "XGBoost": XGBClassifier(random_state = 42)
}


results = {}
for name, model in models.items():
    pipeline = imbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('clf', model)
    ])
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1')
    results[name] = scores.mean()

# Ordonner et afficher resultats
results = {k: v for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True)}
results

{'Naive Bayes': np.float64(0.9834749034749034),
 'Decision Tree': np.float64(0.9831746031746033),
 'Gradient Boosting': np.float64(0.9831746031746033),
 'SVM': np.float64(0.9831746031746033),
 'Logistic Regression': np.float64(0.9777691977691978),
 'Random Forest': np.float64(0.9777691977691978),
 'KNN': np.float64(0.9777691977691978),
 'XGBoost': np.float64(0.9777691977691978)}

# **GridSearchCV**

Une méthode pour trouver automatiquement les meilleurs hyperparamètres d’un modèle, en testant toutes les combinaisons possibles dans une “grille” (grid) de valeurs.

In [50]:
from sklearn.model_selection import GridSearchCV

best_model = DecisionTreeClassifier()
pipeline = imbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('clf', best_model)
    ])

param_grid = {
    'clf__criterion': ['gini', 'entropy'],
    'clf__max_depth': [None, 5, 10, 15],
    'clf__min_samples_split': [2, 3, 4]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1')
grid_search.fit(X_train, y_train)

print("Meilleur score F1:", grid_search.best_score_)
print("Meilleurs hyperparamètres:", grid_search.best_params_)

Meilleur score F1: 0.9831746031746033
Meilleurs hyperparamètres: {'clf__criterion': 'gini', 'clf__max_depth': None, 'clf__min_samples_split': 2}
